# Qwen3-8B Self-Distillation (Iter 2) — Colab Inference Notebook

Self-contained notebook for testing the **LoRA causal adapter** trained on GSM8K + MATH-500 via self-distillation (iteration 2) on Google Colab.

Model location: `causal_models/qwen3_8b_iter2/` in Google Drive.

## Loading strategy

1. Load **Qwen3-8B base** from HuggingFace in **4-bit** (NF4 via `bitsandbytes`).
2. Read `adapter_config.json` from the adapter folder to confirm the base model id and LoRA config.
3. Load the **PEFT adapter** on top via `PeftModel.from_pretrained(base, adapter_path)`.
4. Override the tokenizer's chat template with the **exact** `chat_template.jinja` saved alongside the adapter — no guessing.

**Base vs adapter comparison**: the base model is just `Qwen/Qwen3-8B` loaded in 4-bit without the PEFT adapter applied. Same prompt, same decoding — the delta is purely the self-distillation adapter's effect on GSM8K and MATH-500 reasoning.

## What to put on Google Drive

| Path (in Drive) | What it is |
|---|---|
| `MyDrive/causal_models/qwen3_8b_iter2/` | LoRA adapter folder (contains `adapter_config.json`, `adapter_model.safetensors`, `chat_template.jinja`, tokenizer files) |
| `MyDrive/sample_test.jsonl` | sample test problems from GSM8K + MATH-500 |

Edit the paths in the **Config** cell if your Drive layout is different.

## Runtime

**Runtime → Change runtime type → T4 GPU** (or any GPU). Qwen3-8B in 4-bit fits in ~5-6 GB VRAM, well within T4's 16 GB. Base + adapter active uses ~6-7 GB.

## 1 · Install dependencies

`peft` for adapter loading, `bitsandbytes` for 4-bit quantization, `accelerate` for `device_map="auto"`.

In [ ]:
!pip install -q --upgrade transformers accelerate peft bitsandbytes sentencepiece

## 2 · Config — paths in Google Drive

Mounts Drive and points at the adapter folder + sample data. The base model name is **not** hardcoded — it's read from `adapter_config.json` in the next cell.

In [ ]:
import os, json

# Mount Drive (the adapter folder lives here).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print(f"(Not on Colab or Drive mount skipped: {e})")
    DRIVE_ROOT = "."

ADAPTER_PATH     = os.path.join(DRIVE_ROOT, "causal_models/qwen3_8b_iter2")  # PEFT/LoRA adapter folder (self-distillation iter2)
SAMPLE_DATA_PATH = os.path.join(DRIVE_ROOT, "sample_test.jsonl")             # test problems from GSM8K + MATH-500

# Sanity-check that the adapter folder has the required files.
required = ["adapter_config.json", "chat_template.jinja"]
for name in required:
    p = os.path.join(ADAPTER_PATH, name)
    assert os.path.exists(p), f"Missing: {p}"

print(f"Adapter dir : {ADAPTER_PATH}")
print(f"Sample data : {SAMPLE_DATA_PATH}  (exists: {os.path.exists(SAMPLE_DATA_PATH)})")
print(f"Contents    : {sorted(os.listdir(ADAPTER_PATH))}")

## 3 · Read `adapter_config.json` and `chat_template.jinja`

Both come straight from the adapter folder — base model name and chat template are not guessed.

In [ ]:
with open(os.path.join(ADAPTER_PATH, "adapter_config.json"), encoding="utf-8") as f:
    ADAPTER_CFG = json.load(f)

BASE_MODEL_NAME = ADAPTER_CFG["base_model_name_or_path"]

with open(os.path.join(ADAPTER_PATH, "chat_template.jinja"), encoding="utf-8") as f:
    CHAT_TEMPLATE = f.read()

print(f"Base model (from adapter_config.json): {BASE_MODEL_NAME}")
print()
print("LoRA config:")
for k in ("peft_type", "r", "lora_alpha", "lora_dropout", "target_modules", "bias", "task_type"):
    if k in ADAPTER_CFG:
        print(f"  {k}: {ADAPTER_CFG[k]}")

print(f"\nchat_template.jinja: {len(CHAT_TEMPLATE)} chars")
print("--- first 400 chars ---")
print(CHAT_TEMPLATE[:400])
print("...")

## 4 · Imports & 4-bit quantization config

NF4 with bf16 compute (or fp16 on T4 if bf16 is unsupported) — standard QLoRA-style inference setup.

In [ ]:
import re, time, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "4-bit loading via bitsandbytes requires a CUDA GPU. Switch runtime to GPU."

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = COMPUTE_DTYPE,
)

# Mirrors sft/eval_correct.DATASET_MAX_TOKENS.
MAX_NEW_TOKENS = {"gsm8k": 2048, "math500": 8192, "external": 4096}

print(f"GPU: {torch.cuda.get_device_name(0)} | compute dtype: {COMPUTE_DTYPE}")

## 5 · Answer-extraction helpers

Copied verbatim from `algo/equivalent_ans.py`. Local-only grading — no LLM judge.

In [ ]:
def _extract_boxed(text: str) -> str:
    """Last \\boxed{...}; handles nested braces."""
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1:
            break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                if depth == 0:
                    results.append(text[idx + 7:i])
                    start = i + 1
                    break
                depth -= 1
        else:
            break
    return results[-1].strip() if results else ""


def _normalize(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|\\degree|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\dfrac", r"\\frac", t)
    t = re.sub(r"\\tfrac", r"\\frac", t)
    t = re.sub(r"\$+", "", t)
    t = re.sub(r"\s+", "", t)
    return t.lower()


def is_correct_local(extracted: str, ground_truth: str) -> bool:
    if not extracted:
        return False
    if _normalize(extracted) == _normalize(ground_truth):
        return True
    try:
        return float(extracted.replace(",", "").strip()) == float(str(ground_truth).replace(",", "").strip())
    except (ValueError, TypeError):
        return False


def extract_answer(text: str, dataset: str) -> str:
    boxed = _extract_boxed(text)
    if boxed:
        return boxed
    if dataset == "gsm8k":
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m:
            return m.group(1).replace(",", "").strip()
    return ""


def strip_think(text: str) -> str:
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text

## 6 · Tokenizer + prompt builder

Tokenizer is loaded from the **adapter folder** (training saves it there alongside the adapter, so it has the right special-token state). Its chat template is then explicitly overwritten with the contents of `chat_template.jinja` to guarantee we use the exact template the adapter was trained with — not whatever default ships with the tokenizer.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-pad for generation (matches sft/eval_correct.py)
tokenizer.chat_template = CHAT_TEMPLATE  # use the exact template saved with the adapter


def build_prompt(tok, question: str) -> str:
    msgs = [{"role": "user", "content": question.strip()}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tok.apply_chat_template(msgs, enable_thinking=True, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)


# Sanity check the prompt format.
_sample_prompt = build_prompt(tokenizer, "What is 2 + 2?")
print("-- sample prompt (first 500 chars) --")
print(_sample_prompt[:500])
print("-- end --")
print(f"prompt length: {len(_sample_prompt)} chars")

## 7 · Load base model (4-bit) + apply PEFT adapter

Base is `BASE_MODEL_NAME` (read from `adapter_config.json`) downloaded directly from the HF Hub. The PEFT adapter is then applied with `PeftModel.from_pretrained` — no merging, no full-model save needed.

In [ ]:
def load_base_4bit():
    print(f"Downloading & loading base in 4-bit: {BASE_MODEL_NAME}")
    mdl = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=BNB_CONFIG,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager",
    )
    mdl.eval()
    print(f"  VRAM after base load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    return mdl


def load_adapter_on(base):
    print(f"Applying PEFT adapter: {ADAPTER_PATH}")
    mdl = PeftModel.from_pretrained(base, ADAPTER_PATH)
    mdl.eval()
    print(f"  VRAM after adapter:   {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    return mdl


_base = load_base_4bit()
adapter_model = load_adapter_on(_base)

## 8 · Load sample test data

JSONL with `{"question", "answer", "dataset"}` per line — same schema as `data/gsm8k/test.jsonl` and `data/MATH-500/test.jsonl`.

In [ ]:
with open(SAMPLE_DATA_PATH, encoding="utf-8") as f:
    SAMPLES = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(SAMPLES)} samples")
for i, s in enumerate(SAMPLES, 1):
    print(f"  [{i}] {s.get('dataset','?'):7s}  ans={str(s.get('answer',''))[:30]!r:32s}  q[:60]={s['question'][:60]!r}")

## 9 · Generation + display helpers

In [ ]:
@torch.no_grad()
def generate_one(model, tok, question: str, dataset: str) -> dict:
    prompt = build_prompt(tok, question)
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    max_new = MAX_NEW_TOKENS.get(dataset, MAX_NEW_TOKENS["external"])
    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tok.eos_token_id,
    )
    in_len = enc["input_ids"].shape[1]
    new_toks = out[0][in_len:]
    response = tok.decode(new_toks, skip_special_tokens=True)

    return {
        "prompt":      prompt,
        "response":    response,
        "extracted":   extract_answer(response, dataset),
        "answer_tail": strip_think(response),
        "new_tokens":  int(len(new_toks)),
        "seconds":     time.time() - t0,
    }


def show_result(idx, ex, result, model_label):
    sep = "=" * 78
    print(sep)
    print(f"#{idx:02d}  [{ex.get('dataset','external')}]  model: {model_label}")
    print(sep)
    print("QUESTION:")
    q = ex["question"]
    print(q[:600] + ("..." if len(q) > 600 else ""))
    print()
    expected = ex.get("answer", "")
    print(f"EXPECTED ANSWER : {expected!r}")
    print(f"MODEL EXTRACTED : {result['extracted']!r}")
    if expected:
        print(f"CORRECT         : {is_correct_local(result['extracted'], expected)}")
    print(f"TOKENS / TIME   : {result['new_tokens']} tok / {result['seconds']:.1f}s")
    print("-" * 78)
    print("MODEL OUTPUT (post-</think> tail):")
    print(result["answer_tail"][:400] or "(empty — </think> never closed)")
    print("-- full response excerpt --")
    print(result["response"][:800].rstrip() + ("..." if len(result["response"]) > 800 else ""))
    print()

## 10 · Run the LoRA adapter on the sample test data

In [ ]:
adapter_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(adapter_model, tokenizer, ex["question"], ex.get("dataset", "external"))
    adapter_results.append((ex, r))
    show_result(i, ex, r, "ADAPTER (Qwen3-8B + causal LoRA, self-distillation iter2)")

adapter_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in adapter_results if e.get("answer"))
adapter_graded  = sum(1 for e, _ in adapter_results if e.get("answer"))
print(f"\nSELF-DISTILLATION (Iter2) accuracy on GSM8K/MATH-500: {adapter_correct}/{adapter_graded} = {adapter_correct/max(adapter_graded,1)*100:.1f}%")

## 11 · Base model vs adapter — side-by-side

The base model is **just** `Qwen/Qwen3-8B` (whatever `BASE_MODEL_NAME` resolved to) loaded in 4-bit, **without** the PEFT adapter on top. Same prompt format, same decoding settings, same answer extraction — the only difference is the adapter.

The simplest way to compare is to call `adapter_model.disable_adapter()` as a context manager — this hides the LoRA weights for the duration of the block, so we don't need to load Qwen3-8B a second time (which would OOM on T4).

In [ ]:
base_results = []
with adapter_model.disable_adapter():
    for i, ex in enumerate(SAMPLES, 1):
        r = generate_one(adapter_model, tokenizer, ex["question"], ex.get("dataset", "external"))
        base_results.append((ex, r))
        show_result(i, ex, r, "BASE (Qwen3-8B, no adapter — baseline)")

base_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in base_results if e.get("answer"))
base_graded  = sum(1 for e, _ in base_results if e.get("answer"))
print(f"\nBASE (no adapter) accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%")

In [ ]:
# Comparison table: Base vs Self-Distillation Adapter
row_fmt = "{:<3} {:<8} {:<22} {:<22} {:<22} {:<7} {:<7} {:<7} {:<7}"
print(row_fmt.format("#", "dataset", "expected", "base", "adapter(iter2)", "base_ok", "adp_ok", "b_tok", "a_tok"))
print("-" * 116)
tok_base = tok_adp = 0
for i, ((eb, rb), (ea, ra)) in enumerate(zip(base_results, adapter_results), 1):
    bok = is_correct_local(rb["extracted"], eb["answer"]) if eb.get("answer") else False
    aok = is_correct_local(ra["extracted"], ea["answer"]) if ea.get("answer") else False
    tok_base += rb["new_tokens"]
    tok_adp  += ra["new_tokens"]
    print(row_fmt.format(
        i, eb.get("dataset", ""),
        (str(eb.get("answer") or ""))[:21],
        (rb["extracted"]         or "")[:21],
        (ra["extracted"]         or "")[:21],
        "Y" if bok else "N",
        "Y" if aok else "N",
        rb["new_tokens"],
        ra["new_tokens"],
    ))
n = max(len(SAMPLES), 1)
print("-" * 116)
print(f"BASE           accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%  avg tokens: {tok_base/n:.0f}")
print(f"ADAPTER (Iter2) accuracy: {adapter_correct}/{adapter_graded} = {adapter_correct/max(adapter_graded,1)*100:.1f}%  avg tokens: {tok_adp/n:.0f}")
print(f"\nSELF-DISTILLATION DELTA  : {(adapter_correct-base_correct)/n*100:+.1f} pp accuracy, {(tok_adp-tok_base)/n:+.0f} tokens avg")

## 12 · External test data (CSV / JSONL)

If the teacher uploads their own test file, point this section at it. Schemas:

* **JSONL** — one JSON object per line. Required: `question` (alias: `problem`, `prompt`). Optional: `answer` (alias: `ground_truth`, `answerKey`) and `dataset` (`gsm8k` / `math500` / default `external`).
* **CSV** — same column names. Without an `answer` column, results print without a correctness flag.

Aliases match the keys `sft/eval_correct.py` recognises.

In [ ]:
QUESTION_KEYS = ("question", "problem", "prompt")
ANSWER_KEYS   = ("answer", "ground_truth", "answerKey")


def _pick(record, keys):
    for k in keys:
        if k in record and record[k] not in (None, ""):
            return record[k]
    return None


def load_external(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith(".jsonl"):
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
    elif path.lower().endswith(".csv"):
        import csv
        with open(path, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        raise ValueError(f"Unsupported extension: {path} (need .jsonl or .csv)")

    examples = []
    for r in rows:
        q = _pick(r, QUESTION_KEYS)
        if not q:
            continue
        examples.append({
            "question": str(q),
            "answer":   str(_pick(r, ANSWER_KEYS) or ""),
            "dataset":  str(r.get("dataset", "external")).lower(),
        })
    return examples


def run_external(path, model, tok, model_label="ADAPTER", limit=None):
    examples = load_external(path)
    if limit:
        examples = examples[:limit]
    print(f"Loaded {len(examples)} examples from {path}")
    out = []
    correct = n_graded = 0
    for i, ex in enumerate(examples, 1):
        r = generate_one(model, tok, ex["question"], ex["dataset"])
        show_result(i, ex, r, model_label)
        out.append({**ex, **r})
        if ex["answer"]:
            n_graded += 1
            correct  += int(is_correct_local(r["extracted"], ex["answer"]))
    if n_graded:
        print(f"\nAccuracy: {correct}/{n_graded} = {correct/n_graded*100:.1f}%")
    else:
        print("\nNo ground-truth answers in file — accuracy not computed.")
    return out


# EXTERNAL_PATH = "/content/drive/MyDrive/teacher_test.jsonl"   # or .csv
# external_outputs = run_external(EXTERNAL_PATH, adapter_model, tokenizer, model_label="ADAPTER", limit=20)

### Optional — save external-run outputs to a file

In [ ]:
# OUTPUT_PATH = "inference_external_results.jsonl"
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     for row in external_outputs:
#         f.write(json.dumps({
#             "question":         row["question"],
#             "ground_truth":     row["answer"],
#             "extracted_answer": row["extracted"],
#             "model_answer":     row["response"],
#             "new_tokens":       row["new_tokens"],
#             "seconds":          row["seconds"],
#             "dataset":          row["dataset"],
#         }, ensure_ascii=False) + "\n")
# from google.colab import files
# files.download(OUTPUT_PATH)